# Day 2 | Cleaning and preprocessing

This notebook is the source of truth for cleaning. It reads the confidential raw HR source, audits repeated IDs, cleans names and dates in Python, applies documented rules, validates results, and writes the interim CSV. Employee_Name is normalized and split; DOB is parsed and used to derive age at hire. Names, exact DOBs, and source IDs are removed before export.

## 1. Load data and duplicate audit

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd, numpy as np, re
RAW=Path("../data/raw/HRDataset_EDA_raw.csv")
AS_OF_DATE=pd.Timestamp("2026-10-29")
df=pd.read_csv(RAW,dtype={"EmpID":"string"}).rename(columns={'EmpID': 'employee_id', 'Employee_Name': 'employee_name', 'DOB': 'date_of_birth', 'Sex': 'sex', 'MaritalDesc': 'marital_status', 'Department': 'department', 'Position': 'job_title', 'RecruitmentSource': 'recruitment_source', 'DateofHire': 'hire_date', 'DateofTermination': 'termination_date', 'TermReason': 'termination_reason', 'EmploymentStatus': 'employment_status', 'Termd': 'terminated_source', 'Salary': 'salary', 'PerformanceScore': 'performance_score', 'EngagementSurvey': 'engagement_score', 'EmpSatisfaction': 'employee_satisfaction', 'Absences': 'absences', 'DaysLateLast30': 'days_late_last_30', 'WorkMode': 'work_mode'})
audit=pd.read_csv("../data/interim/duplicate_employee_audit.csv")
print("Raw shape",df.shape,"audit excess duplicate rows",audit.excess_rows.sum())
print("Conflicting duplicate groups",audit.conflict_columns.ne("No conflict").sum())
assert audit.conflict_columns.fillna("No conflict").eq("No conflict").all()


## 2. Normalize whitespace and deduplicate only after audit

In [ ]:
for col in df.select_dtypes(include="string").columns:
    df[col]=df[col].str.replace(r"\s+"," ",regex=True).str.strip()
df["duplicate_employee_flag"]=df.duplicated("employee_id",keep="first")
def title_name_token(token):
    token=token.title()
    # Restore common internal capitalization after title-casing (e.g. McDonald).
    return re.sub(r"\bMc([a-z])",lambda m:"Mc"+m.group(1).upper(),token)

def standardize_employee_name(value):
    if pd.isna(value): return pd.NA
    tokens=re.sub(r"\s+"," ",str(value)).strip().split(" ")
    return " ".join(title_name_token(token) for token in tokens)

df["employee_name"]=df.employee_name.map(standardize_employee_name).astype("string")
print("Duplicate rows removed",int(df.duplicate_employee_flag.sum()))
df=df.loc[~df.duplicate_employee_flag].copy()
assert len(df)==5500 and df.employee_id.is_unique


## 3. Parse Employee_Name for the cleaning workflow

The source stores names as `Last First` with an optional trailing middle initial. Whitespace and case are standardized before parsing. Name parts are useful for the cleaning exercise, but all identity fields are dropped before either cleaned CSV is written.

In [ ]:
def split_employee_name(value):
    if pd.isna(value) or not str(value).strip():
        return pd.Series([pd.NA,pd.NA,pd.NA])
    parts=str(value).split()
    last=parts[0] if len(parts)>=1 else pd.NA
    first=parts[1] if len(parts)>=2 else pd.NA
    remainder=parts[2:]
    middle=(remainder[-1].rstrip(".").upper() if remainder and len(remainder[-1].rstrip("."))==1 else pd.NA)
    return pd.Series([last,first,middle])

df[["last_name","first_name","middle_initial"]]=df.employee_name.apply(split_employee_name)
print("Names normalized and split; identity fields are withheld from displays and exports.")
assert df.employee_name.notna().all()


## 4. Canonical category mappings

In [ ]:
df["sex"]=df.sex.str.casefold().map({"f":"Female","female":"Female","m":"Male","male":"Male"}).fillna("Missing")
df["marital_status"]=df.marital_status.str.casefold().replace({"marred":"married"}).map({"single":"Single","married":"Married","divorced":"Divorced","separated":"Separated","widowed":"Widowed"}).fillna("Missing")
dep={"it/is":"IT/IS","it":"IT/IS","production":"Production","sales":"Sales","software engineering":"Software Engineering","software eng.":"Software Engineering","admin offices":"Admin Office","admin office":"Admin Office","executive office":"Executive Office"}
df["department"]=df.department.str.casefold().map(dep)
pos={"accountant i":"Accountant I","administrative assistant":"Administrative Assistant","area sales manager":"Area Sales Manager","bi developer":"BI Developer","bi director":"BI Director","cio":"Chief Information Officer (CIO)","data analyst":"Data Analyst","data architect":"Data Architect","database administrator":"Database Administrator","director of operations":"Director of Operations","director of sales":"Director of Sales","enterprise architect":"Enterprise Architect","it director":"IT Director","it manager - db":"IT Manager - Database","it manager - infra":"IT Manager - Infrastructure","it manager - support":"IT Manager - Support","it support":"IT Support","network engineer":"Network Engineer","president & ceo":"President & CEO","principal data architect":"Principal Data Architect","production manager":"Production Manager","production technician i":"Production Technician I","production technician ii":"Production Technician II","sales manager":"Sales Manager","sr. accountant":"Senior Accountant","senior accountant":"Senior Accountant","senior bi developer":"Senior BI Developer","sr. dba":"Senior Database Administrator","sr. network engineer":"Senior Network Engineer","shared services manager":"Shared Services Manager","software engineer":"Software Engineer","software engineering manager":"Software Engineering Manager"}
df["job_title"]=df.job_title.str.casefold().map(pos)
executive={"Chief Information Officer (CIO)","President & CEO","IT Director","Director of Operations","Director of Sales","BI Director"}
df["position_group"]=np.where(df.job_title.isin(executive),"Executive",df.job_title)
r=df.recruitment_source.str.casefold().replace({"indeed.com":"indeed","linked in":"linkedin","on-line web application":"online web application"})
df["recruitment_source"]=r.map({"linkedin":"LinkedIn","indeed":"Indeed","google search":"Google Search","employee referral":"Employee Referral","diversity job fair":"Diversity Job Fair","careerbuilder":"Career Builder","website":"Website","online web application":"Online Web Application","other":"Other","unknown":"Unknown"}).fillna("Unknown")
df["employment_status"]=df.employment_status.str.casefold().map({"active":"Active","voluntarily terminated":"Voluntarily Terminated","terminated for cause":"Terminated for Cause"})
df["work_mode"]=df.work_mode.str.casefold().map({"remote":"Remote","onsite":"Onsite","hybrid":"Hybrid"}).fillna("Missing")
df["performance_score"]=df.performance_score.str.casefold().map({"fully meets":"Fully Meets","exceeds":"Exceeds","needs improvement":"Needs Improvement","pip":"PIP"}).fillna("Missing")
reason_map={"another position":"Another Position","unhappy":"Unhappy","hours":"Hours","more money":"More Money","career change":"Career Change","attendance":"Attendance","relocation out of area":"Relocation Out Of Area","performance":"Performance","medical issues":"Medical Issues","no-call no-show":"No-Call No-Show","return to school":"Return To School","military":"Military","gross misconduct":"Gross Misconduct","retiring":"Retiring","maternity leave - did not return":"Maternity Leave - Did Not Return","n/a-stillemployed":"N/A - Still Employed","n/a - still employed":"N/A - Still Employed"}
rr=df.termination_reason.str.casefold().str.replace(r"\s+"," ",regex=True).str.strip()
df["termination_reason"]=rr.map(reason_map).fillna("Unknown")
df["termination_reason_conflict_flag"]=(df.employment_status.ne("Active")&df.termination_reason.eq("N/A - Still Employed"))
df.loc[df.termination_reason_conflict_flag,"termination_reason"]="Unknown"


## 5. Parse dates and resolve chronology conservatively

In [ ]:
def parse_date(v,col):
    if pd.isna(v): return pd.NaT
    v=str(v).strip()
    fmts=(['%m/%d/%y','%m/%d/%Y'] if '/' in v else (['%d-%b-%y','%d-%B-%y'] if re.search('[A-Za-z]',v) else ['%d-%m-%Y','%d-%m-%y']))
    for fmt in fmts:
        try:
            dt=pd.Timestamp(datetime.strptime(v,fmt))
            if col in ["hire_date","termination_date"] and '%y' in fmt and dt.year<2000: dt=dt+pd.DateOffset(years=100)
            return dt
        except ValueError: pass
    return pd.NaT
termination_source=df.termination_date.copy()
for col in ["hire_date","termination_date"]: df[col]=pd.to_datetime([parse_date(v,col) for v in df[col]])
def age_on_date(birth_date, reference_date):
    if pd.isna(birth_date) or pd.isna(reference_date): return np.nan
    return reference_date.year-birth_date.year-((reference_date.month,reference_date.day)<(birth_date.month,birth_date.day))

def parse_dob(value, hire_date):
    if pd.isna(value): return pd.NaT
    value=str(value).strip()
    try:
        if "/" in value:
            date_format="%m/%d/%y" if len(value.rsplit("/",1)[-1])==2 else "%m/%d/%Y"
        else:
            date_format="%d-%m-%Y"  # dash dates are interpreted as day-month-year per the supplied convention
        dob=pd.Timestamp(datetime.strptime(value,date_format))
    except ValueError:
        return pd.NaT
    # Correct future DOBs and any DOB that would place the employee under 18 at hire.
    if dob>AS_OF_DATE or (pd.notna(hire_date) and age_on_date(dob,hire_date)<18):
        dob=dob-pd.DateOffset(years=100)
    return dob

dob_raw=df.date_of_birth.copy()
df["dob_missing_flag"]=dob_raw.isna() | dob_raw.astype("string").str.strip().eq("").fillna(False)
df["dob_dash_day_first_assumption_flag"]=dob_raw.astype("string").str.contains("-",na=False)
df["date_of_birth"]=pd.to_datetime([parse_dob(v,h) for v,h in zip(dob_raw,df.hire_date)])
df["dob_parse_failed_flag"]=~df.dob_missing_flag & df.date_of_birth.isna()
df["age_at_hire"]=pd.array([age_on_date(d,h) for d,h in zip(df.date_of_birth,df.hire_date)],dtype="Int64")
df["age_at_hire_outside_18_70_flag"]=df.age_at_hire.notna() & ~df.age_at_hire.between(18,70)
print("Expected missing DOB",int(df.dob_missing_flag.sum()),"unparsed nonblank DOB",int(df.dob_parse_failed_flag.sum()),"age outside 18–70",int(df.age_at_hire_outside_18_70_flag.sum()),"dash dates using day-first assumption",int(df.dob_dash_day_first_assumption_flag.sum()))

df["future_hire_flag"]=df.hire_date>AS_OF_DATE;df["future_termination_flag"]=df.termination_date>AS_OF_DATE
df.loc[df.future_hire_flag,"hire_date"]=pd.NaT;df.loc[df.future_termination_flag,"termination_date"]=pd.NaT
df["date_conflict_flag"]=False
# Try day/month swap only for conflicting numeric dash termination dates; otherwise retain null and flag.
for ix in df.index[df.termination_date.notna()&(df.termination_date<df.hire_date)]:
    v=str(termination_source.loc[ix])
    # Parsed date is already a timestamp, so no guessed swap; leave unresolved rather than invent source text.
    df.loc[ix,"date_conflict_flag"]=True;df.loc[ix,"termination_date"]=pd.NaT
df["active_had_termination_date_flag"]=df.employment_status.eq("Active")&df.termination_date.notna()
df.loc[df.employment_status.eq("Active"),"termination_date"]=pd.NaT


## 6. Convert numeric columns and flag questionable values

In [ ]:
def to_num(s): return pd.to_numeric(s.astype("string").str.replace("$","",regex=False).str.replace(" ","",regex=False).str.replace(",","",regex=False),errors="coerce")
df["salary"]=to_num(df.salary);salary_before_correction=df.salary.copy();df["salary_corrected_flag"]=False
for ix,v in df.salary.items():
    if pd.isna(v) or 20000<=v<=300000: continue
    cand=v*10 if v<20000 else v/10
    peers=df.loc[(df.job_title==df.loc[ix,"job_title"])&df.salary.between(20000,300000),"salary"]
    if len(peers)>=5 and peers.quantile(.01)<=cand<=peers.quantile(.99): df.loc[ix,"salary"]=cand;df.loc[ix,"salary_corrected_flag"]=True
    else: df.loc[ix,"salary"]=np.nan
df["engagement_score"]=pd.to_numeric(df.engagement_score.astype("string").str.replace(" ",".",regex=False),errors="coerce")
df["engagement_corrected_flag"]=df.engagement_score.isin([41,45]);df.loc[df.engagement_corrected_flag,"engagement_score"]/=10
df["engagement_invalid_flag"]=~df.engagement_score.between(1,5);df.loc[df.engagement_invalid_flag,"engagement_score"]=np.nan
df["employee_satisfaction"]=pd.to_numeric(df.employee_satisfaction,errors="coerce").astype("Int64").where(lambda x:x.between(1,5))
absn=to_num(df.absences);df["absences_negative_flag"]=absn<0;df["absences_over_60_flag"]=absn>60;df["absences"]=absn.mask(absn<0).astype("Int64")
late=to_num(df.days_late_last_30);df["days_late_invalid_flag"]=late>30;df["days_late_last_30"]=late.mask(late>30).astype("Int64");df["is_ever_late"]=(df.days_late_last_30.fillna(0)>0).astype("int8")
# Employment status is source of truth; drop redundant raw status flag from analysis data.
status_raw=to_num(df.terminated_source);# Aggregate before/after check without exporting original salary values.
corrected=df.salary_corrected_flag
if corrected.any():
    print("Corrected salary rows",int(corrected.sum()),"median before",salary_before_correction[corrected].median(),"median after",df.loc[corrected,"salary"].median())
df["terminated_status_conflict_flag"]=status_raw.ne(df.employment_status.ne("Active").astype(int));df["terminated"]=df.employment_status.ne("Active").astype("int8");df=df.drop(columns=["terminated_source"])


## 7. Engineer consistent time features and validate

In [ ]:
df["hire_year"]=df.hire_date.dt.year
df["hire_month"]=df.hire_date.dt.month
df["hire_quarter"]=df.hire_date.dt.quarter
df["tenure_days"]=((df.termination_date.fillna(AS_OF_DATE)-df.hire_date).dt.days)
df["tenure_years"]=(df.tenure_days/365.2425).round(2)
assert len(df)==5500 and df.employee_id.is_unique
assert df.age_at_hire.dropna().ge(18).all()
assert df.engagement_score.dropna().between(1,5).all()
assert df.absences.dropna().ge(0).all() and df.days_late_last_30.dropna().between(0,30).all()
assert not (df.hire_date>AS_OF_DATE).any() and not (df.termination_date>AS_OF_DATE).any()
assert not (df.termination_date.notna()&(df.termination_date<df.hire_date)).any()
assert not (df.employment_status.eq("Active")&df.termination_date.notna()).any()
n_unique_ids=df.employee_id.nunique()
# Only necessary analysis columns and flags are exported; no raw copies, names, exact DOBs, or source IDs; derived age_at_hire is retained.
df=df.drop(columns=["duplicate_employee_flag","employee_id","employee_name","last_name","first_name","middle_initial","date_of_birth"])
df.to_csv("../data/interim/cleaned_day2.csv",index=False)
print("PASS",df.shape,"unique IDs checked before export",n_unique_ids)
print("Absences >60 retained and flagged:",int(df.absences_over_60_flag.sum()),"salary corrections",int(df.salary_corrected_flag.sum()))
